In [ ]:
%pip install matplotlib pandas numpy scikit-learn  mlflow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow

In [ ]:
pd.set_option('display.max_columns', None) # display all columns

df = pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')
df.head()

In [ ]:
df.isna().sum().sum()

In [ ]:
X = df.drop(columns=['Diabetes_binary'])
y = df['Diabetes_binary']

In [ ]:
mlflow.set_experiment('diabetes_classification')

### Using LogisticRegression model with Cross-Validation and feature selection

In [ ]:
mlflow.start_run(run_name='LogisticRegression_final')
mlflow.set_tag("Feature Selection", True)
mlflow.set_tag("Scaled Data", True)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scaler = StandardScaler()

# X_train_scaled = scaler.fit_transform(X_train)

# X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

sfm = SelectFromModel(model)
sfm.fit(X_train, y_train)

selected_features = X.columns[sfm.get_support()]
SFM_df = df[selected_features]
SFM_df

splitting and scaling the selected features

In [ ]:
X = SFM_df
y = df['Diabetes_binary']

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

logreg = LogisticRegression(max_iter=1, fit_intercept=False, random_state=42, n_jobs=-1)

# parameter grid
param_grid = {
    "penalty": ["l1", "l2", "elasticnet"],
    "C": [100000,10000000],
    "solver": ["saga"],
    "l1_ratio": [0.5],
}

# Set up Stratified K-Folds for cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define the GridSearchCV object
grid_search = GridSearchCV(
    estimator=logreg,
    param_grid=param_grid,
    cv=cv,
    verbose=2, # Show progress of GridSearch
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

In [ ]:
# Log parameters, metrics, and the best model
mlflow.log_param("param_grid", param_grid)
mlflow.log_metric("accuracy", grid_search.best_score_)

for param, value in grid_search.best_params_.items():
    mlflow.log_param(param, value)

# Log the best model
best_model = grid_search.best_estimator_
mlflow.sklearn.log_model(best_model, "LogisticRegression_final")

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

y_pred = best_model.predict(X_test)

# metrics
accuracy = accuracy_score(y_test, y_pred)
class_report = classification_report(y_test, y_pred)
presicion, recall, f1 = precision_score(y_test, y_pred), recall_score(y_test, y_pred), f1_score(y_test, y_pred)

# log metrics
mlflow.log_metric("accuracy", accuracy)
mlflow.log_metric("presicion", presicion)
mlflow.log_metric("recall", recall)
mlflow.log_metric("f1_score", f1)

# get confusion matrix values
conf_matrix = confusion_matrix(y_test,y_pred)
true_positive = conf_matrix[0][0]
true_negative = conf_matrix[1][1]
false_positive = conf_matrix[0][1]
false_negative = conf_matrix[1][0]

# log confusion matrix values
mlflow.log_metric("true_positive", true_positive)
mlflow.log_metric("true_negative", true_negative)
mlflow.log_metric("false_positive", false_positive)
mlflow.log_metric("false_negative", false_negative)

# Save confusion matrix plot to a file
plt.figure()
ConfusionMatrixDisplay(conf_matrix).plot()
plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png')
# plt.close()

# Log the saved plot as an artifact
mlflow.log_artifact("confusion_matrix.png","confusion_matrix")

print(f"Presicion: {presicion}")
print(f"Recall: {recall}")
print(f"Accuracy: {accuracy}")
print('-'*50)
print(f"Classification Report:\n{class_report}")
print('-'*50)
plt.show()
mlflow.end_run()

Rule Based and LogisticRegression

In [ ]:
import mlflow
uri = "mlruns/157510782472443811/9a8e3ebd197844e4b90eea99377cac6a/artifacts/LogisticRegression_final"
model = mlflow.sklearn.load_model(uri)
model_prediction = model.predict(X_test)

In [ ]:
from sklearn.model_selection import train_test_split

df = pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')

X = df.drop(columns=['Diabetes_binary'])
y = df['Diabetes_binary']

_, X_test, _, y_test = train_test_split(X, y, test_size=0.0001, random_state=311111)

selected_features = ['HighBP', 'HighChol', 'CholCheck', 'HvyAlcoholConsump']

def predict_diabetes(row):
    score = 0

    if row["HighBP"] == 1.0:
        score += 1
    if row["HighChol"] == 1.0:
        score += 1
    if row["BMI"] > 40.0:
        score += 1
    if row["PhysActivity"] == 0.0:
        score += 1
    if row["Smoker"] == 1.0:
        score += 1
    if row["Age"] >= 8:
        score += 1
    if row["DiffWalk"] == 1.0:
        score += 1
    if row["GenHlth"] >= 4.0:  # Poor general health
        score += 1

    # Predict diabetes if score exceeds a threshold otherwise use the built ML model
    return 1 if score >= 5 else best_model.predict(row[selected_features].values.reshape(1, -1))[0]

# Apply the rule-based prediction
rule_based_predict = X_test.apply(predict_diabetes, axis=1)

# Create a DataFrame to display results
# results_df = pd.DataFrame({
#     'Rule-based Prediction': rule_based_predict,
#     'Actual': y_test
# })

from sklearn.metrics import accuracy_score

r2_rule_based = accuracy_score(y_test, rule_based_predict)

print(f"Rule-based Prediction Accuracy: {r2_rule_based}")
print('\n')

In [ ]:
mlflow.end_run()

# XGBClassifier

In [ ]:
mlflow.set_experiment('diabetes_classification')
mlflow.start_run(run_name='XGBClasifier_Balanced')
mlflow.set_tag("Feature Selection", True)

In [ ]:
%pip install xgboost

In [ ]:
from sklearn.model_selection import train_test_split

df = pd.read_csv('diabetes_binary_5050split_health_indicators_BRFSS2015.csv')

X = df.drop(columns=['Diabetes_binary'])
y = df['Diabetes_binary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.feature_selection import SelectFromModel
from xgboost import XGBClassifier

# Fit an XGBClassifier on the training data
xgb_fs = XGBClassifier(objective='binary:logistic', use_label_encoder=False, eval_metric='logloss')
xgb_fs.fit(X_train, y_train)

# Use feature importance for selection; here we use the median importance as threshold
selector = SelectFromModel(xgb_fs, threshold='median', prefit=True)

# Get the selected feature names
selected_features = X_train.columns[selector.get_support()]
print("Selected features:", selected_features)

# Transform the training and test sets to use only the selected features
X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)

In [ ]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 4, 6],
    'learning_rate': [0.01, 0.1, 1],
    'subsample': [0.8, 1.0],
}

xgb = XGBClassifier(objective='binary:logistic', use_label_encoder=False, eval_metric='logloss')

grid_search = GridSearchCV(
    estimator=xgb, 
    param_grid=param_grid, 
    scoring='accuracy', 
    cv=5, 
    n_jobs=-1, 
    verbose=1
)
grid_search.fit(X_train_selected, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)

# Evaluate on the test set using selected features
best_xgb = grid_search.best_estimator_
test_accuracy = best_xgb.score(X_test_selected, y_test)
print("Test set accuracy:", test_accuracy)


In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
y_pred_xgb = best_xgb.predict(X_test_selected)

# compute metrics
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
class_report_xgb = classification_report(y_test, y_pred_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb)
recall_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)

# log metrics to mlflow
mlflow.log_metric("accuracy", accuracy_xgb)
mlflow.log_metric("precision", precision_xgb)
mlflow.log_metric("recall", recall_xgb)
mlflow.log_metric("f1_score", f1_xgb)

# compute and log confusion matrix details
cm = confusion_matrix(y_test, y_pred_xgb)
tp = cm[0][0]
tn = cm[1][1]
fp = cm[0][1]
fn = cm[1][0]

mlflow.log_metric("true_positive", tp)
mlflow.log_metric("true_negative", tn)
mlflow.log_metric("false_positive", fp)
mlflow.log_metric("false_negative", fn)

# plot and log confusion matrix
plt.figure()
ConfusionMatrixDisplay(cm).plot()
plt.title('XGBClassifier Confusion Matrix')
plt.savefig('xgb_confusion_matrix.png')
mlflow.log_artifact('xgb_confusion_matrix.png', "xgb_confusion_matrix")

# print the metrics
print(f"XGB Precision: {precision_xgb}")
print(f"XGB Recall: {recall_xgb}")
print(f"XGB Accuracy: {accuracy_xgb}")
print('-'*50)
print(f"XGB Classification Report:\n{class_report_xgb}")
plt.show()

# log the best model
mlflow.sklearn.log_model(best_xgb, "xgb_classifier_balanced")
mlflow.end_run()


In [ ]:
!mlflow ui --port 5000